In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Data Cleaning/personal_expense_classification.csv")
print(df.head())

  expense_id  amount     merchant      description       category
0       EXP1   78.04       Amazon    fuel purchase       shopping
1       EXP2  190.39  Apple Store  online shopping     technology
2       EXP3  147.74   McDonald's     app purchase           food
3       EXP4  121.74    Starbucks             meal           food
4       EXP5   35.42      Netflix    fuel purchase  entertainment


In [28]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.duplicated().sum())

(100, 5)
expense_id      object
amount         float64
merchant        object
description     object
category        object
dtype: object
expense_id     0
amount         0
merchant       0
description    0
category       0
dtype: int64
0


In [29]:
for col in ["merchant", "description", "category"]:
    print(col)
    print(df[col].value_counts())
    print()

merchant
merchant
Amazon         21
McDonald's     15
Starbucks      15
Shell          15
Netflix        12
Apple Store    11
Uber           11
Name: count, dtype: int64

description
description
ride to work            21
online shopping         19
fuel purchase           14
app purchase            13
meal                    12
coffee purchase         11
monthly subscription    10
Name: count, dtype: int64

category
category
food             30
transport        26
shopping         21
entertainment    12
technology       11
Name: count, dtype: int64



In [30]:
print(df["amount"].describe())
print("Zero or negative amounts:", (df["amount"] <= 0).sum())

count    100.000000
mean      96.685600
std       58.010059
min        6.080000
25%       42.677500
50%       95.505000
75%      147.387500
max      197.440000
Name: amount, dtype: float64
Zero or negative amounts: 0


In [31]:
q1 = df["amount"].quantile(0.25)
q3 = df["amount"].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

outliers = df[(df["amount"] < low) | (df["amount"] > high)]
print("Outliers found:", len(outliers))

Outliers found: 0


In [32]:
print(pd.crosstab(df["merchant"], df["category"]))
print(pd.crosstab(df["merchant"], df["description"]))

category     entertainment  food  shopping  technology  transport
merchant                                                         
Amazon                   0     0        21           0          0
Apple Store              0     0         0          11          0
McDonald's               0    15         0           0          0
Netflix                 12     0         0           0          0
Shell                    0     0         0           0         15
Starbucks                0    15         0           0          0
Uber                     0     0         0           0         11
description  app purchase  coffee purchase  fuel purchase  meal  \
merchant                                                          
Amazon                  4                2              2     3   
Apple Store             2                1              0     3   
McDonald's              2                1              3     0   
Netflix                 1                0              2     1   
Shel

In [33]:
clean = df.copy()   # keep the original safe

# 1. Remove the ID column (it has no predictive value)
clean = clean.drop(columns=["expense_id"])

# 2. Standardise text (good habit even when data looks fine)
for col in ["merchant", "description", "category"]:
    clean[col] = clean[col].str.strip().str.lower()

# 3. Optional: remove description because it does not match merchant
# clean = clean.drop(columns=["description"])

In [34]:
print(clean.shape)
print(clean.head())
print(clean.isna().sum())

(100, 4)
   amount     merchant      description       category
0   78.04       amazon    fuel purchase       shopping
1  190.39  apple store  online shopping     technology
2  147.74   mcdonald's     app purchase           food
3  121.74    starbucks             meal           food
4   35.42      netflix    fuel purchase  entertainment
amount         0
merchant       0
description    0
category       0
dtype: int64


In [35]:
clean.to_csv("/content/drive/MyDrive/Data Cleaning/personal_expense_classification_clean.csv", index=False)

In [36]:
clean = clean.drop(columns=["description"])

print(clean.shape)
print(clean.head())

clean.to_csv("/content/drive/MyDrive/Data Cleaning/personal_expense_classification_clean_v2.csv", index=False)

(100, 3)
   amount     merchant       category
0   78.04       amazon       shopping
1  190.39  apple store     technology
2  147.74   mcdonald's           food
3  121.74    starbucks           food
4   35.42      netflix  entertainment


In [37]:
print(pd.crosstab(clean["merchant"], clean["category"]))

category     entertainment  food  shopping  technology  transport
merchant                                                         
amazon                   0     0        21           0          0
apple store              0     0         0          11          0
mcdonald's               0    15         0           0          0
netflix                 12     0         0           0          0
shell                    0     0         0           0         15
starbucks                0    15         0           0          0
uber                     0     0         0           0         11


In [38]:
print(clean.groupby("category")["amount"].mean())

category
entertainment    118.689167
food              93.548667
shopping         105.945238
technology       100.230000
transport         81.171154
Name: amount, dtype: float64


In [39]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Data Cleaning/my_personal_expense_data_1_clean.csv",
                 parse_dates=["Date"])
print(df.dtypes)

Date              datetime64[ns]
Account                   object
Category                  object
Note                      object
INR                      float64
Income/Expense            object
dtype: object


In [40]:
exp = df[df["Income/Expense"] == "expense"].copy()
print(exp.shape)

(231, 6)


In [41]:
transfer = (
    (exp["Category"] == "other") &
    exp["Note"].str.contains(r"^(?:to|sent to|gave to) |lended|returned|recharge by|for company", regex=True)
)
print(transfer.sum())
exp = exp[~transfer].copy()
print(exp.shape)

21
(210, 6)


In [42]:
exp["Note"] = exp["Note"].replace({"snack": "snacks"})
exp["Note"] = exp["Note"].str.replace("barath", "bharath")

In [43]:
counts = exp["Category"].value_counts()
rare = counts[counts < 5].index.tolist()
print(rare)    # education, self-development, beauty, gift
exp["Category"] = exp["Category"].replace(rare, "other")
print(exp["Category"].value_counts())

['education', 'other', 'self-development', 'beauty', 'gift']
Category
food              156
transportation     31
apparel             7
household           6
social life         5
other               5
Name: count, dtype: int64


In [44]:
exp = exp.drop(columns=["Note"])

In [45]:
print(exp.shape)
print(exp.isna().sum())
exp.to_csv("/content/drive/MyDrive/Data Cleaning/my_personal_expense_data_1_clean_v2.csv", index=False)

(210, 5)
Date              0
Account           0
Category          0
INR               0
Income/Expense    0
dtype: int64
